# v5 — valuing a ball in play as events

v4 modelled a swing as what it produces — whiff, foul, or ball in play — and
valued a ball in play with a regression of its run value. This notebook values
a ball in play as one of six events instead:

```
E[value | in play, s] = Σ P(event | in play, s) · RE(event, count)
```

over single, double, triple, home run, out and error, with run values from each
fold's table. The count stays a feature — it changes the odds of each event, as
hitters change their approach — while the table supplies what each event is
worth in that count. Triples and errors are expected to be predicted near their
base rates; that is a stated limitation, not a failure.

Then it builds the project's two outputs on the result, and runs the
contamination screens and benchmarks on them.

**Only the in-play branch changes.** The take model and the whiff / foul /
in-play classifier are fitted identically in both variants, and the notebook
asserts their predictions are identical.

**The selection rule, fixed before the numbers:** paired held-out MSE on all
swings, classifier against regression, 95% interval from resampling games. The
classifier wins if the interval excludes zero in its favour; otherwise the
regression stays. Calibration then checks the winner; the player checks are
guardrails.

Held out on the personalized folds, 2023–2025. 2026 is not loaded.

In [1]:
import sys; sys.path.insert(0, '..')
import numpy as np, pandas as pd

from src import data as D, evaluate as E, features as F, decision as DEC
from src.baselines import fit_direct, predict_both, V1_FEATURES
from src.decomposition import fit_decomposed

pd.set_option('display.width', 180); pd.set_option('display.max_columns', 30)
SEASONS = range(2021, 2026)          # 2026 is kept out of model selection; not loaded here
PF = E.PERSONALIZED_FOLDS
SHOW, MODEL = E.SHOW, E.MODEL_COLS
ACC_COLS = ['pitches', 'diff_%', 'diff_%_lo', 'diff_%_hi', 'verdict']
V3_FEATURES = ['plate_x_bat','plate_z_norm','count','stand','p_throws',
               'release_speed','pfx_x','pfx_z','pitch_type']
PRIORS = ['prior_whiff', 'prior_foul']

def row(run, label, score=DEC.DEFAULT_SCORE):
    return E.harness(run, label, score)['summary']

def paired(a, b, labels):
    return E.paired_accuracy(a, b, labels)[ACC_COLS]

def rounds(run):
    return pd.DataFrame({season: m.rounds for season, m in run.models.items()}).T

In [2]:
df = D.drop_pitchers_batting(D.load_seasons(SEASONS, verbose=False))
df = D.add_zone_frame(df.dropna(subset=['plate_x_mid','plate_z_mid']))
for c in ('stand','p_throws','pitch_type'):
    df[c] = df[c].astype('category')

# Hitter priors, each season from the two seasons before it (2021 is prior data only).
key = ['game_pk', 'at_bat_number', 'pitch_number']
pdf = F.season_hot_zone(df).merge(F.season_contact_priors(df)[key + PRIORS],
                                  on=key, how='left', validate='one_to_one')
assert pdf[PRIORS].notna().all().all()
print(f'{len(pdf):,} pitches, seasons {sorted(pdf.season.unique())}')

2,827,989 pitches, seasons [np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


## 1. The in-play branch: regression vs classification

In [3]:
FEATURES = {'generic':      V3_FEATURES,                          # no hitter features
            'personalized': V3_FEATURES + ['hot_zone'] + PRIORS}  # + hot zone, contact priors
in_play_runs = {}
for name, feats in FEATURES.items():
    regression_run = E.run_folds(pdf, feats, PF, fit=fit_decomposed)
    classifier_run = E.run_folds(pdf, feats, PF, fit=fit_decomposed, in_play='classifier')
    for c in ['q_take', 'p_whiff', 'p_foul', 'p_bip']:
        assert np.array_equal(regression_run.held[c].to_numpy(), classifier_run.held[c].to_numpy()), f'{name}: {c} differs'
    in_play_runs[name] = {'regression': regression_run, 'classifier': classifier_run}
print('take model and whiff / foul / in-play probabilities identical between variants: yes')
pd.concat({name: rounds(r['classifier']) for name, r in in_play_runs.items()}, axis=1)

take model and whiff / foul / in-play probabilities identical between variants: yes


generic                   personalized                  
        take outcome bip_value         take outcome bip_value
2023     125     305        88          125     375        81
2024     155     535       103          155     527       119
2025     250     578       139          250     668       182

In [4]:
# The deciding test: squared error on the same held-out swings. Negative = classifier better.
inplay_verdict = pd.concat({name: paired(r['classifier'], r['regression'], ('classifier', 'regression'))
                            for name, r in in_play_runs.items()}, names=['output'])
inplay_verdict.round(3)

pitches  diff_%  diff_%_lo  diff_%_hi            verdict
output       action season                                                          
generic      take   2023     374424   0.000      0.000      0.000                tie
                    2024     366046   0.000      0.000      0.000                tie
                    2025     368391   0.000      0.000      0.000                tie
                    pooled  1108861   0.000      0.000      0.000                tie
             swing  2023     339205  -0.007     -0.032      0.021                tie
                    2024     337782  -0.004     -0.025      0.019                tie
                    2025     336884  -0.026     -0.047     -0.001  classifier better
                    pooled  1013871  -0.012     -0.025      0.002                tie
personalized take   2023     374424   0.000      0.000      0.000                tie
                    2024     366046   0.000      0.000      0.000                tie
                    2025     368391   0.000      0.000      0.000                tie
                    pooled  1108861   0.000      0.000      0.000                tie
             swing  2023     339205  -0.090     -0.128     -0.048  classifier better
                    2024     337782  -0.013     -0.041      0.021                tie
                    2025     336884  -0.010     -0.035      0.022                tie
                    pooled  1013871  -0.038     -0.056     -0.018  classifier better

In [5]:
# The rule, applied: the classifier wins only where its pooled swing interval excludes zero in its favour.
pooled_swing = inplay_verdict.xs(('swing', 'pooled'), level=['action', 'season'])
rule = {name: ('classifier' if pooled_swing.loc[name, 'diff_%_hi'] < 0 else 'regression')
        for name in in_play_runs}
print('selection rule alone:', rule)

# Decision: the classifier for both outputs. Personalized, by the rule. Generic
# ties on accuracy, and is switched too so the two outputs differ only in their
# hitter features -- a deliberate override of the rule's tie-keeps-current default.
IN_PLAY = {name: 'classifier' for name in in_play_runs}
IN_PLAY

selection rule alone: {'generic': 'regression', 'personalized': 'classifier'}


{'generic': 'classifier', 'personalized': 'classifier'}

In [6]:
# The branch on its own: in-play value against what happened, on balls in play only.
pd.concat({name: E.paired_accuracy(r['classifier'], r['regression'], ('classifier', 'regression'),
                                   rows='in_play')[ACC_COLS]
           for name, r in in_play_runs.items()}, names=['output']).round(3)

pitches  diff_%  diff_%_lo  diff_%_hi            verdict
output       action  season                                                          
generic      in play 2023     123523  -0.000     -0.059      0.061                tie
                     2024     123279   0.017     -0.041      0.069                tie
                     2025     124070  -0.058     -0.108     -0.003  classifier better
                     pooled   370872  -0.014     -0.048      0.021                tie
personalized in play 2023     123523  -0.186     -0.280     -0.098  classifier better
                     2024     123279  -0.025     -0.093      0.049                tie
                     2025     124070  -0.022     -0.078      0.046                tie
                     pooled   370872  -0.078     -0.122     -0.032  classifier better

In [7]:
pd.concat({(name, variant): E.calibration_table(run).xs('swing', level='action')
           for name, r in in_play_runs.items() for variant, run in r.items()},
          names=['output', 'in-play value']).round(3)

slope  slope_lo  slope_hi
output       in-play value season                           
generic      regression    2023    1.087     1.063     1.112
                           2024    1.008     0.986     1.031
                           2025    1.038     1.015     1.061
             classifier    2023    1.063     1.040     1.084
                           2024    0.983     0.961     1.005
                           2025    1.004     0.983     1.027
personalized regression    2023    0.936     0.913     0.958
                           2024    0.966     0.946     0.987
                           2025    0.978     0.958     1.001
             classifier    2023    0.980     0.960     1.002
                           2024    0.935     0.915     0.954
                           2025    0.954     0.934     0.975

In [8]:
# What the in-play classifier gets right, per event, held out.
pd.concat({name: E.in_play_diagnostics(r['classifier'].held) for name, r in in_play_runs.items()},
          names=['output']).round(4)

balls in play  observed rate  mean predicted  predicted spread (sd)  calibration slope  logloss improvement % (all events)
output       season event                                                                                                                                  
generic      2023   single              123523         0.2096          0.2086                 0.0378             0.7619                              1.5920
                    double              123523         0.0662          0.0644                 0.0208             1.0400                              1.5920
                    triple              123523         0.0057          0.0051                 0.0034             0.6375                              1.5920
                    home_run            123523         0.0473          0.0420                 0.0228             1.2464                              1.5920
                    field_out           123523         0.6627          0.6708                 0.0440             0.8130                              1.5920
                    field_error         123523         0.0085          0.0090                 0.0047             0.4581                              1.5920
             2024   single              123279         0.2084          0.2101                 0.0361             0.8818                              1.8144
                    double              123279         0.0627          0.0646                 0.0212             0.9964                              1.8144
                    triple              123279         0.0056          0.0055                 0.0032             0.8344                              1.8144
                    home_run            123279         0.0441          0.0441                 0.0249             1.1191                              1.8144
                    field_out           123279         0.6703          0.6668                 0.0444             0.8332                              1.8144
                    field_error         123279         0.0090          0.0088                 0.0043             0.6574                              1.8144
             2025   single              124070         0.2092          0.2093                 0.0374             0.8434                              1.7208
                    double              124070         0.0621          0.0641                 0.0217             0.8909                              1.7208
                    triple              124070         0.0051          0.0056                 0.0034             0.7459                              1.7208
                    home_run            124070         0.0454          0.0446                 0.0262             1.0818                              1.7208
                    field_out           124070         0.6701          0.6678                 0.0452             0.8408                              1.7208
                    field_error         124070         0.0081          0.0087                 0.0043             0.4914                              1.7208
personalized 2023   single              123523         0.2096          0.2092                 0.0379             0.7717                              1.7454
                    double              123523         0.0662          0.0646                 0.0222             0.9584                              1.7454
                    triple              123523         0.0057          0.0050                 0.0033             0.7239                              1.7454
                    home_run            123523         0.0473          0.0426                 0.0279             1.1294                              1.7454
                    field_out           123523         0.6627          0.6696                 0.0506             0.7559                              1.7454
                    field_error         123523         0.0085          0.0090                 0.0045             0.4292              

In [9]:
# What personalization does to the event odds: personalized minus generic event
# probabilities on the same pitches, by the hitter's hot-zone value at the pitch.
events = ['single', 'double', 'triple', 'home_run', 'field_out', 'field_error']
g = in_play_runs['generic']['classifier'].held
pz = in_play_runs['personalized']['classifier'].held
m = g[E.PITCH_KEY + [f'p_{e}' for e in events]].merge(
    pz[E.PITCH_KEY + ['hot_zone'] + [f'p_{e}' for e in events]], on=E.PITCH_KEY, suffixes=('_g', '_p'))
m['hot zone quintile'] = pd.qcut(m['hot_zone'], 5, labels=['lowest', '2', '3', '4', 'highest'])
shift = pd.DataFrame({e: m[f'p_{e}_p'] - m[f'p_{e}_g'] for e in events}).groupby(m['hot zone quintile'], observed=True).mean()
(shift * 100).round(2)          # percentage points

,single,double,triple,home_run,field_out,field_error
hot zone quintile,,,,,,
lowest,0.47,-0.35,-0.01,-0.69,0.62,-0.04
2,-0.08,-0.25,-0.00,-0.50,0.82,0.01
3,-0.06,-0.29,0.01,-0.37,0.70,0.01
4,-0.23,0.14,0.00,-0.11,0.20,-0.01
highest,-0.14,0.73,-0.05,1.70,-2.24,-0.00


In [10]:
# Guardrails, classifier minus regression, hitter-resampled intervals.
pd.concat({name: E.metric_differences(r['classifier'], r['regression'], labels=('classifier', 'regression'))
           for name, r in in_play_runs.items()}, names=['output']).round(3)

classifier  regression   diff  diff_lo  diff_hi
output       check                                                                   
generic      chase | zone_swing           -0.910      -0.914  0.004    0.003    0.005
             zone_swing | chase            0.689       0.706 -0.017   -0.021   -0.014
             YoY R2 (mean)                 0.583       0.583 -0.000   -0.005    0.005
             Zone% |r|                     0.276       0.270  0.006    0.004    0.009
             next-season r (partial)       0.094       0.100 -0.005   -0.008   -0.002
personalized chase | zone_swing           -0.864      -0.862 -0.003   -0.005   -0.001
             zone_swing | chase            0.576       0.582 -0.006   -0.010   -0.002
             YoY R2 (mean)                 0.606       0.615 -0.009   -0.015   -0.003
             Zone% |r|                     0.198       0.184  0.014    0.009    0.018
             next-season r (partial)       0.159       0.173 -0.014   -0.020   -0.008

**By the rule, the classifier wins for the personalized output and ties for the
generic one.** Personalized: swing MSE 0.038% lower (interval 0.018–0.056%).
Generic: 0.012% lower, with an interval that includes zero (−0.025% to
+0.002%). On balls in play alone, where the change is undiluted by whiffs and
fouls, the personalized classifier is 0.078% better (0.032–0.122%) and the
generic one ties.

**The personalized gain is concentrated in 2023** (−0.090%), whose fold trains
on 2022 alone; 2024 and 2025 are ties. Calibration is mixed: the classifier's
swing slopes are closer to 1 for the generic output (1.063, 0.983, 1.004 against
1.087, 1.008, 1.038) but, for the personalized one, better in 2023 and slightly
more over-spread in 2024–25 (0.935 and 0.954 against 0.966 and 0.978).

**What the classifier learns.** Against the base rates it improves log loss by
1.6–1.8% (generic) and 1.7–2.1% (personalized). Without hitter features,
home-run odds vary *more* than it predicts (calibration slopes 1.08–1.25); with
them, close to calibrated (0.99–1.13) — what a hitter does with contact is what
separates home runs. Singles and outs vary less than predicted (0.76–0.88), and
triples and errors stay near their base rates, as expected. Personalization moves the odds the way it
should: for pitches in a hitter's hottest zones it raises P(home run) by 1.7
percentage points and P(double) by 0.7, and lowers P(out) by 2.2; in his
coldest zones it does the reverse.

**The guardrails move slightly the wrong way** — for the personalized output
YoY R² 0.615 → 0.606, Zone% 0.184 → 0.198, next-season 0.173 → 0.159, all
outside their intervals. They flag, they do not decide.

**Decision: the classifier for both outputs.** The personalized output by the
rule. The generic output ties on accuracy and is switched too, so the two
outputs differ only in their hitter features — a deliberate override of the
rule's tie-keeps-current default, recorded as such.

## 2. The two outputs

- **generic** — "was this a good decision for a typical hitter?" No hitter
  features.
- **personalized** — "was this a good decision for *this* hitter?" The hot zone
  in the in-play classifier, the whiff and foul priors in the outcome
  classifier.

Both use the in-play classifier (section 1). **Calibration step:** where a slope
interval excludes 1, each season is recalibrated with a linear map learned one
season earlier, and the map is kept only if it brings the 2024–25 swing slopes
closer to 1 without making held-out MSE worse.

In [11]:
outputs_fitted = {name: in_play_runs[name][IN_PLAY[name]] for name in in_play_runs}
print('in-play valuation:', IN_PLAY)
recal, maps = {}, {}
for name, run in outputs_fitted.items():
    recal[name], maps[name] = E.recalibrate_last_season(run)
pd.concat(maps, names=['output']).round(4)

in-play valuation: {'generic': 'classifier', 'personalized': 'classifier'}


fitted on       a       b
output       season action                           
generic      2024   take         2023  0.0008  1.0005
                    swing        2023  0.0054  1.0588
             2025   take         2024 -0.0002  0.9984
                    swing        2024 -0.0010  0.9809
personalized 2024   take         2023  0.0008  1.0005
                    swing        2023  0.0022  0.9759
             2025   take         2024 -0.0002  0.9984
                    swing        2024 -0.0030  0.9337

In [12]:
cal = pd.concat({(name, version): E.calibration_table(r)
                 for name in outputs_fitted
                 for version, r in [('as fitted', outputs_fitted[name]), ('recalibrated', recal[name])]},
                names=['output', 'version'])
cal.round(3)

slope  slope_lo  slope_hi
output       version      action season                           
generic      as fitted    take   2023    1.000     0.996     1.003
                                 2024    0.998     0.994     1.002
                                 2025    0.994     0.991     0.998
                          swing  2023    1.063     1.040     1.084
                                 2024    0.983     0.961     1.005
                                 2025    1.004     0.983     1.027
             recalibrated take   2023    1.000     0.996     1.003
                                 2024    0.998     0.994     1.001
                                 2025    0.996     0.992     0.999
                          swing  2023    1.063     1.040     1.084
                                 2024    0.929     0.908     0.949
                                 2025    1.024     1.002     1.047
personalized as fitted    take   2023    1.000     0.996     1.003
                                 2024    0.998     0.994     1.002
                                 2025    0.994     0.991     0.998
                          swing  2023    0.980     0.960     1.002
                                 2024    0.935     0.915     0.954
                                 2025    0.954     0.934     0.975
             recalibrated take   2023    1.000     0.996     1.003
                                 2024    0.998     0.994     1.001
                                 2025    0.996     0.992     0.999
                          swing  2023    0.980     0.960     1.002
                                 2024    0.958     0.937     0.977
                                 2025    1.021     1.001     1.045

In [13]:
# Keep the map only if 2024-25 slopes move closer to 1 and MSE is not worse there.
def distance_from_1(table):
    t = table.xs('swing', level='action')
    return float((t.loc[[2024, 2025], 'slope'] - 1).abs().mean())

decision = {}
for name in outputs_fitted:
    acc = E.paired_accuracy(recal[name], outputs_fitted[name], ('recalibrated', 'as fitted'))
    acc = acc[acc.index.get_level_values('season').isin([2024, 2025])]
    not_worse = bool((acc['diff_lo'] <= 0).all())
    closer = distance_from_1(E.calibration_table(recal[name])) < distance_from_1(E.calibration_table(outputs_fitted[name]))
    decision[name] = closer and not_worse
    print(f"{name:13s} swing slopes closer to 1: {closer}   MSE not worse: {not_worse}   -> keep map: {decision[name]}")
    display(acc[ACC_COLS].round(3))
outputs = {name: (recal[name] if decision[name] else outputs_fitted[name]) for name in outputs_fitted}

generic       swing slopes closer to 1: False   MSE not worse: False   -> keep map: False


pitches  diff_%  diff_%_lo  diff_%_hi           verdict
action season                                                         
take   2024     366046   0.062      0.044      0.079  as fitted better
       2025     368391   0.047      0.043      0.051  as fitted better
swing  2024     337782   0.037      0.024      0.049  as fitted better
       2025     336884   0.002     -0.001      0.005               tie

personalized  swing slopes closer to 1: True   MSE not worse: False   -> keep map: False


pitches  diff_%  diff_%_lo  diff_%_hi           verdict
action season                                                         
take   2024     366046   0.062      0.044      0.079  as fitted better
       2025     368391   0.047      0.043      0.051  as fitted better
swing  2024     337782   0.011      0.005      0.017  as fitted better
       2025     336884  -0.005     -0.014      0.004               tie

In [14]:
final = pd.DataFrame([row(outputs[name], name) for name in outputs]).set_index('variant')
final[MODEL + SHOW]

,swing MSE vs count-only %,swing pred slope,take MSE vs count-only %,take pred slope,chase | zone_swing,zone_swing | chase,split-half r,YoY R2 (mean),Zone% |r|,next-season r (partial)
variant,,,,,,,,,,
generic,1.96,1.017,70.08,0.997,-0.910,0.689,0.816,0.583,0.276,0.094
personalized,2.12,0.956,70.08,0.997,-0.864,0.576,0.844,0.606,0.198,0.159


In [15]:
# Does Q_swing hold where it is extrapolated? Swings at pitches the league usually takes.
p_swing = E.swing_propensity(pdf, V3_FEATURES, PF)
pd.concat({name: E.propensity_calibration(outputs[name], p_swing) for name in outputs},
          names=['output']).round(4)

pitches  mean pred  mean obs   slope  slope_lo  slope_hi
output       action league swing propensity                                                          
generic      swing  (-0.001, 0.1]              13481    -0.0618   -0.0666  0.9554    0.9023    1.0176
                    (0.1, 0.3]                 70228    -0.0521   -0.0515  1.0275    0.9970    1.0556
                    (0.3, 0.7]                356086    -0.0315   -0.0313  1.0232    1.0025    1.0414
                    (0.7, 0.9]                322603    -0.0221   -0.0200  1.0032    0.9720    1.0313
                    (0.9, 1.0]                251473    -0.0025   -0.0005  0.9982    0.9594    1.0250
             take   (-0.001, 0.1]             363234     0.0530    0.0536  0.9922    0.9879    0.9964
                    (0.1, 0.3]                290224     0.0281    0.0292  0.9961    0.9927    0.9998
                    (0.3, 0.7]                361843     0.0002    0.0010  1.0001    0.9970    1.0033
                    (0.7, 0.9]                 81067    -0.0395   -0.0381  1.0123    1.0028    1.0203
                    (0.9, 1.0]                 12493    -0.1354   -0.1339  1.0360    1.0206    1.0497
personalized swing  (-0.001, 0.1]              13481    -0.0619   -0.0666  0.9228    0.8774    0.9962
                    (0.1, 0.3]                 70228    -0.0520   -0.0515  1.0201    0.9883    1.0481
                    (0.3, 0.7]                356086    -0.0310   -0.0313  0.9815    0.9590    1.0015
                    (0.7, 0.9]                322603    -0.0209   -0.0200  0.9219    0.8958    0.9489
                    (0.9, 1.0]                251473     0.0000   -0.0005  0.9183    0.8833    0.9465
             take   (-0.001, 0.1]             363234     0.0530    0.0536  0.9922    0.9879    0.9964
                    (0.1, 0.3]                290224     0.0281    0.0292  0.9961    0.9927    0.9998
                    (0.3, 0.7]                361843     0.0002    0.0010  1.0001    0.9970    1.0033
                    (0.7, 0.9]                 81067    -0.0395   -0.0381  1.0123    1.0028    1.0203
                    (0.9, 1.0]                 12493    -0.1354   -0.1339  1.0360    1.0206    1.0497

In [16]:
# How different are the two answers?
sg, sp = (E.player_metric(outputs[n].held, 'signed_edge') for n in ('generic', 'personalized'))
m = sg.merge(sp, on=['season', 'batter'], suffixes=('_g', '_p'))
print(f"generic vs personalized hitter-season scores: r = {np.corrcoef(m.decision_value_g, m.decision_value_p)[0, 1]:.3f}")
try:
    from pybaseball import playerid_reverse_lookup
    s25 = m[m.season == 2025].copy()
    nm = playerid_reverse_lookup(s25.batter.tolist())
    nm['name'] = (nm.name_first + ' ' + nm.name_last).str.title()
    s25 = s25.merge(nm[['key_mlbam', 'name']], left_on='batter', right_on='key_mlbam')
    s25['personalized - generic'] = s25.decision_value_p - s25.decision_value_g
    cols = ['name', 'pitches_g', 'decision_value_g', 'decision_value_p', 'personalized - generic']
    for title, frame in [('TOP 10, 2025 (generic)', s25.nlargest(10, 'decision_value_g')),
                         ('TOP 10, 2025 (personalized)', s25.nlargest(10, 'decision_value_p')),
                         ('BOTTOM 10, 2025 (generic)', s25.nsmallest(10, 'decision_value_g')),
                         ('LARGEST GAP, personalized above generic', s25.nlargest(8, 'personalized - generic')),
                         ('LARGEST GAP, personalized below generic', s25.nsmallest(8, 'personalized - generic'))]:
        print(title); display(frame[cols].round(1))
except Exception as exc:
    print(f'name lookup unavailable ({exc})')

generic vs personalized hitter-season scores: r = 0.944


Gathering player lookup table. This may take a moment.


TOP 10, 2025 (generic)


,name,pitches_g,decision_value_g,decision_value_p,personalized - generic
142,Leo Rivas,520,132.2,127.1,-5.1
280,Davis Schneider,1012,128.5,124.4,-4.1
116,Gleyber Torres,2652,126.7,125.6,-1.2
301,Max Schuemann,838,125.1,121.6,-3.5
97,Matt Thaiss,768,124.5,120.2,-4.3
31,Max Muncy,1603,123.7,124.4,0.7
139,Ronald Acuña,1741,123.7,131.8,8.2
154,Kyle Tucker,2275,120.5,124.6,4.1
357,Dylan Beavers,615,120.2,116.7,-3.5
223,Justyn-Henry Malloy,550,119.2,114.9,-4.3


TOP 10, 2025 (personalized)


...

,name,pitches_g,decision_value_g,decision_value_p,personalized - generic
47,Javier Báez,1543,62.0,65.7,3.7
296,Jhonkensy Noel,563,70.3,68.9,-1.4
83,Edmundo Sosa,942,71.1,69.9,-1.2
256,Michael Harris,2173,71.9,80.6,8.7
268,Lenyn Sosa,2056,72.9,76.6,3.7
326,Angel Martínez,1793,74.8,74.7,-0.0
0,Martín Maldonado,569,75.1,73.9,-1.2
286,Ronny Mauricio,692,75.4,74.3,-1.2
120,Yohel Pozo,591,77.5,74.3,-3.2
354,Joey Ortiz,1873,78.4,81.8,3.5


LARGEST GAP, personalized above generic


,name,pitches_g,decision_value_g,decision_value_p,personalized - generic
65,Corey Seager,1668,109.0,119.9,10.9
256,Michael Harris,2173,71.9,80.6,8.7
291,Bobby Witt,2511,98.2,106.8,8.5
139,Ronald Acuña,1741,123.7,131.8,8.2
179,Vladimir Guerrero,2502,104.6,112.7,8.1
269,Yainer Diaz,1871,79.0,87.1,8.1
118,Yandy Díaz,2520,106.8,114.6,7.8
138,Shohei Ohtani,2830,104.7,112.5,7.8


LARGEST GAP, personalized below generic


,name,pitches_g,decision_value_g,decision_value_p,personalized - generic
229,Bryce Teodosio,521,102.5,94.4,-8.1
266,Alexander Canario,897,100.2,93.2,-7.0
48,Austin Hedges,696,100.2,93.3,-6.9
315,Brooks Baldwin,1251,95.4,89.0,-6.4
314,Connor Norby,1245,102.5,96.2,-6.3
259,Gabriel Arias,1792,97.8,92.1,-5.7
343,Isaac Collins,1792,117.6,112.1,-5.5
32,Michael Taylor,1267,105.5,100.0,-5.5


**Neither output is recalibrated.** The last-season map makes held-out MSE
worse for both (takes by about 0.05–0.06%, swings by up to 0.037% in 2024), so
both are used as fitted:

- **generic:** swing calibration slopes 0.98–1.06, take 0.99–1.00;
- **personalized:** swing 0.94–0.98 — slightly over-spread, most on swings at
  pitches the league usually swings at — take 0.99–1.00.

**Both are a little too generous to rare swings.** On swings at pitches the
league swings at less than 10% of the time they predict about −0.062 runs
against −0.067 observed.

**The two answers mostly agree** — hitter-season scores correlate 0.944 — and
differ where they should. Personalization lifts hitters whose contact does
damage (Seager +10.9, Harris, Witt, Acuña, Guerrero, Diaz, Ohtani about +8) and
lowers hitters whose decisions look better against a typical hitter's values
than against their own (Teodosio, Canario, Hedges, Baldwin, Norby).

### Contamination on the two outputs

Zone% correlated with the score, raw and holding fixed the correct-decision
rate, chase and zone-swing rates, or production. `correct_decision` is shown on
each model for comparison; its correct-decision-rate cell is circular.

In [17]:
rows = []
for name in outputs:
    for score in ['signed_edge', 'correct_decision']:
        rows.append({'output': name, 'score': score, **E.zone_contamination(outputs[name].held, score)})
pd.DataFrame(rows).set_index(['output', 'score']).round(3)

Zone% r raw  Zone% r | correct-decision rate  Zone% r | chase, zone-swing  Zone% r | production  corr(Zone%, production)
output       score                                                                                                                                     
generic      signed_edge             0.276                            0.129                       -0.044                 0.379                   -0.225
             correct_decision        0.245                            0.048                        0.110                 0.339                   -0.225
personalized signed_edge             0.198                           -0.047                       -0.193                 0.331                   -0.225
             correct_decision        0.252                            0.032                        0.067                 0.355                   -0.225

**The same pattern as v4.** Without hitter features `signed_edge` is the more
contaminated score — generic 0.276 raw, 0.379 holding production fixed, against
0.245 and 0.339 for `correct_decision`. With them it is the less contaminated
one — personalized 0.198 and 0.331, against 0.252 and 0.355. As before the
answer depends on the control, so none of this corrects the bias.

## 3. Against the benchmarks

Everything held out, 2023–2025. O-Swing% (negated) and Z-Swing% − O-Swing% are
computed from the same pitches; their construct-validity cells are circular, so
only reliability, contamination and prediction are comparable.

In [18]:
v1_run = E.run_folds(pdf, V1_FEATURES, PF, fit=fit_direct)

h = v1_run.held
inz = D.in_rulebook_zone(h, ball_edge=True)
k_ = ['season', 'batter']
rates = pd.DataFrame({'o_swing': h[~inz].groupby(k_, observed=True).swing.mean(),
                      'z_swing': h[inz].groupby(k_, observed=True).swing.mean(),
                      'pitches': h.groupby(k_, observed=True).size()}).reset_index()
rates = rates[rates.pitches >= E.QUALIFY_PITCHES]

def as_scores(values):
    s = rates[k_ + ['pitches']].assign(raw=values)
    z = s.groupby('season').raw.transform(lambda v: (v - v.mean()) / v.std(ddof=0))
    return s.assign(decision_value=100 + 10 * z)

def checks_from_scores(scores, label):
    return {'variant': label,
            'YoY R2 (mean)': round(E.yoy_reliability(scores).r2.mean(), 3),
            'Zone% |r|': round(E.zone_pct_correlation(h, scores).r.abs().mean(), 3),
            'next-season r (partial)': round(E.predictive_validity(h, scores).r_partial.mean(), 3)}

bench = pd.DataFrame([
    {**E.metric_checks(v1_run.held, 'y_pred')['summary'], 'variant': 'v1 as designed (chosen_value)'},
    row(outputs['generic'], 'generic output'),
    row(outputs['personalized'], 'personalized output'),
    checks_from_scores(as_scores(-rates.o_swing), 'O-Swing% (negated)'),
    checks_from_scores(as_scores(rates.z_swing - rates.o_swing), 'Z-Swing% − O-Swing%'),
]).set_index('variant')
bench[SHOW]

,chase | zone_swing,zone_swing | chase,split-half r,YoY R2 (mean),Zone% |r|,next-season r (partial)
variant,,,,,,
v1 as designed (chosen_value),-0.890,0.678,0.763,0.538,0.116,0.129
generic output,-0.910,0.689,0.816,0.583,0.276,0.094
personalized output,-0.864,0.576,0.844,0.606,0.198,0.159
O-Swing% (negated),NaN,NaN,NaN,0.697,0.421,0.077
Z-Swing% − O-Swing%,NaN,NaN,NaN,0.557,0.114,0.146


## Findings

All figures held out, 2023–2025, on the personalized folds.

**Valuing a ball in play as events is at least as good as regressing its run
value**, and better for the personalized output (swing MSE −0.038%, interval
excluding zero; generic −0.012%, a tie). The gain rests mainly on the 2023
fold, and the guardrails move slightly the wrong way. Both outputs adopt it —
the generic one by decision rather than by the rule, so the two outputs differ
only in their hitter features.

**The two outputs:**

| | swing MSE vs count-only | swing slope | chase\|zs | zs\|chase | split-half | YoY R² | Zone% \|r\| | next-season |
|---|---|---|---|---|---|---|---|---|
| generic | −1.96% | 1.017 | −0.910 | 0.689 | 0.816 | 0.583 | 0.276 | 0.094 |
| personalized | −2.12% | 0.956 | −0.864 | 0.576 | 0.844 | 0.606 | 0.198 | 0.159 |

**The classifier is also easier to read.** Personalization shows up where it
should — a hitter's hot zone raises his home-run and double odds and lowers his
out odds.

**Against simple benchmarks.** O-Swing% is heavily contaminated (Zone% |r|
0.421) and weakly predictive (0.077). Z-Swing% − O-Swing% is less contaminated
than either output (0.114) and nearly as predictive as the personalized one
(0.146 against 0.159), though less reliable (YoY 0.557).

The final models, refitted on every development season and saved for use
outside the notebooks, are built in `final_models.ipynb`.